# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their structure. All dataset entities (record sets, fields, columns, etc.) are referenced **by their `@id` fields**.

In [ ]:
# List all record sets with their @ids, names, and fields
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in this Croissant metadata.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', 'No name')}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}")
            print(f"      Name: {getattr(field, 'name', 'No name')}")
            print(f"      Data type: {getattr(field, 'data_type', 'Unknown')}")
        print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s.

In [ ]:
# Collect all record set @ids
record_sets = list(dataset.record_sets())
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}")
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# If no record sets are found, print a note
if not dataframes:
    print("No record sets were found or loaded from the dataset.")
else:
    # Display columns from the first available record set
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in {first_rs}: {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Here we illustrate common data processing: filtering records, normalizing numeric fields, and grouping/categorizing data using `@id` fields.

> **Change the `record_set_id` and field `@id`s below to correspond to your dataset's available IDs as seen in the previous cell.**


In [ ]:
# Demo: If a record set with at least one numeric column exists, demonstrate filtering and normalization

from IPython.display import display

if not dataframes:
    print("No dataframes available to perform EDA.")
else:
    # Select the first available dataframe
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to find a numeric column
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print(f"No numeric field found in record set {record_set_id} for EDA.")
    else:
        # Filtering: choose a threshold suitable for any numeric field
        threshold = df[numeric_field_id].quantile(0.8) if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} in {record_set_id}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by a categorical field (search for object-type columns besides the numeric field)
        group_field_id = None
        candidate_group_fields = [c for c in filtered_df.columns if c != numeric_field_id]
        for col in candidate_group_fields:
            if pd.api.types.is_object_dtype(filtered_df[col]) and filtered_df[col].nunique() > 1:
                group_field_id = col
                break
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped.head())
        else:
            print("No suitable categorical column found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` or `seaborn`.

> **Modify the column names below to match your dataset, as revealed earlier.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if not dataframes:
    print("No dataframes to plot.")
else:
    # Use the EDA-selected record_set_id and numeric_field_id if present
    df = dataframes[record_set_id]
    if numeric_field_id:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()
    else:
        print("No numeric field found for visualization.")

    # Optionally, visualize relationship between numeric and group/categorical field
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library. We identified available record sets (by `@id`), loaded their data, and performed exploratory analysis on numeric and categorical fields using these `@id` references. Visualizations demonstrated the distribution and relationships between example variables.

For further analysis, reference your dataset schema—explore other record sets, customize filters, and expand visual analytics to suit your research questions. All processing should continue referencing entities by their `@id` from the Croissant schema for reproducibility and consistency.